# Day 7 — Solution: First Contact (worked exemplar)

*Compare structure and reasoning, not your exact numbers — sample periods
differ. Where your reasoning was thinner than this, note it in your error
log: that is the finding.*

## Setup + computations

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 4)
DATA_SOURCE = os.environ.get("QRC_DATA", "real")

from qrc.data import get_prices
from qrc.synth import synthetic_prices

if DATA_SOURCE == "real":
    px = get_prices(["SPY", "QQQ", "GLD"], start="2010-01-01")
else:
    px = synthetic_prices(n_days=3000, n_assets=3, seed=30)
    px.columns = ["SPY", "QQQ", "GLD"]

rets = px.pct_change().dropna()

ann_vol = rets.std() * np.sqrt(252)
growth = (1 + rets).cumprod()
dd = growth / growth.cummax() - 1
print("annualized vol:"); print(ann_vol.round(4))
print("max drawdown:");  print(dd.min().round(4))

growth.plot(title="Growth of $1"); plt.show()

In [ ]:
rets["SPY"].rolling(63).std().mul(np.sqrt(252)).plot(label="SPY", alpha=.7)
rets["QQQ"].rolling(63).std().mul(np.sqrt(252)).plot(label="QQQ", alpha=.7)
rets["GLD"].rolling(63).std().mul(np.sqrt(252)).plot(label="GLD", alpha=.7)
plt.legend(); plt.title("63-day annualized volatility"); plt.show()

print("correlation matrix:"); print(rets.corr().round(3))
worst20 = rets["SPY"].nsmallest(20).index
sub = rets.loc[worst20]
print(f"corr on SPY's worst 20 days: SPY-QQQ {sub['SPY'].corr(sub['QQQ']):.2f}, "
      f"SPY-GLD {sub['SPY'].corr(sub['GLD']):.2f}")

## Exemplar answers (real data, 2010→; yours will differ)

**Q1.** QQQ typically wins raw return (tech-heavy decade), SPY has the
smallest vol, and drawdowns are comparable in depth (2022 hit both; GLD's
worst is usually shallower but longer). "Highest return" ≠ "best
investment": risk-adjusted (return per unit of vol) and drawdown tolerance
matter; QQQ's edge was paid for with deeper intermediate drawdowns. Ranking
without a risk dimension is the first analytical error this course trains
out of you.

**Q2.** SPY: mean daily ≈ 0.04–0.05%, daily std ≈ 1% → ratio ≈ 0.04–0.05.
The average day's edge is ~5% of one day's noise. Implication: months of
performance are statistically meaningless; only years resolve a real edge
(module 04 quantifies exactly how many).

**Q3.** Days above/below the mean: slightly more down days than up days
(returns are left-skewed); the worst single day (−10%+) is typically ~1.5–2×
the best single day (+5–6%). Asymmetry term: **skewness** (module 03). Its
practical meaning: "average return" understates what a bad week feels like.

**Q4.** Volatility is anything but constant: long calm stretches (2013–2019)
punctuated by violent regimes (Aug 2011, Mar 2020, 2022). It *clusters* —
high-vol days follow high-vol days. A "constant volatility" assumption
understates risk in storms and overstates it in calm — and any risk model
built on it will be exactly wrong when it matters (module 09's GARCH exists
for this).

**Q5.** SPY–QQQ correlation is high (~0.8+): they are largely the same bet.
SPY–GLD is near zero on all days. On SPY's worst 20 days, correlations
shift — the interesting result is that SPY–QQC-type equity pairs go to
~1.0 (no diversification inside equities in a crash), while gold's behavior
varies by episode. Diversification measured in calm times overstates the
protection available in crashes — "correlations go to 1 when you need them
not to."

## Bias audit lite (exemplar)

- **Sample period**: 2010→ includes one of history's longest bull markets.
  QQQ's ranking is partly "the decade we chose". A 2000–2010 window would
  invert conclusions (dot-com aftermath). Without testing multiple periods
  (module 13), any ranking is a period-dependent opinion.
- **Universe**: SPY/QQQ/GLD are *survivors and winners by construction* —
  today's famous instruments. A universe chosen from what worked is
  selection bias at the asset level.
- **Costs**: ignored. Mostly harmless for buy-and-hold ETF comparisons, but
  fatal for anything that trades (reversal strategies live inside the
  spread). Note the general rule: the more turnover, the less "ignoring
  costs" is forgivable.

## Reflection (exemplar)

Learned: the shape of real return data — small mean, large vol, clustered
risk, asymmetric tails, conditional correlations. Assumptions made:
constant-ish vol (rolling window choice), stationarity-ish returns, no
costs. Misleading if: period chosen post-hoc, famous assets pre-selected,
calm-period correlations trusted in crises. Next: quantify the
signal-to-noise ratio properly (module 02), test normality (module 03), and
revisit drawdowns with the right statistics.